Importing requried libaries

In [0]:
import sys
sys.path.append('/Workspace/Users/saik84328@gmail.com')

from pyspark.sql import functions as F
from delta.tables import DeltaTable
from SetUp.Config import bronze_schema, silver_schema, gold_schema
from pyspark.sql.types import *
from datetime import datetime
import uuid


In [0]:
start_time = datetime.now()

In [0]:
%run /Workspace/Users/saik84328@gmail.com/DataBricksLearning/AuditData

In [0]:
%run /Workspace/Users/saik84328@gmail.com/DataBricksLearning/ValidationFramework

Handling Schema **Validations**

In [0]:


Claims_schema =F.StructType([
    StructField("Id", StringType(), True),
    StructField("Claimid", StringType(), True),
    StructField("Chargeid", IntegerType(), True),
    StructField("Patientid", StringType(), True),
    StructField("Type", StringType(), True),
    StructField("Amount", DoubleType(), True),
    StructField("Method", StringType(), True),
    StructField("Fromdate",DateType(), True),
    StructField("Todate",DateType(), True),
    StructField("Placeofservice",StringType(), True),
    StructField("Procdurecode",StringType(), True),
    StructField("Modifier1",StringType(), True),
    StructField("Modifier2",StringType(), True),
    StructField("Diagnosisref1",StringType(), True),
    StructField("Diagnosisref2",StringType(), True),
    StructField("Diagnosisref3",StringType(), True),
    StructField("Diagnosisref4",StringType(), True),
    StructField("Units",IntegerType(), True),
    StructField("Departmentid",StringType(), True),
    StructField("Notes",StringType(), True)  
   
])

Reading Source File

In [0]:


df_raw = (
    spark.read.format("csv")
    .option("header", "true")
    .schema(Claims_schema)
    .load("/Volumes/helathcare_bronze/default/helathcare/claims_transactions.csv")

    .withColumn("Readtimestamp", F.current_timestamp())

    .withColumn("Filename", F.col("_metadata.file_name"))

    .withColumn("file_size", F.col("_metadata.file_size"))
)

display(df_raw)

Creating Bronze Table

In [0]:
df_raw.printSchema()

In [0]:
df_raw.write.format("delta").option("delta.enableChangeDataFeed","true").mode("overwrite").saveAsTable(f"helathcare_bronze.{bronze_schema}.ClaimsData")

In [0]:
%sql
SELECT distinct Method FROM `helathcare_bronze`.`helathcare_bronze`.`ClaimsData` --where Method is not null;

Handling Nulls

In [0]:
#Checking for null values in the bronze table
silver_df = spark.table("helathcare_bronze.helathcare_bronze.ClaimsData")

#conclusion 
#Amount need fill null values with 0
#Method need fill null values with Cash

Here handling column wise Null values

In [0]:
validation_result = run_validations(
    silver_df,
    ["Patientid"]
)

print(f"Duplicate Count : {validation_result['duplicates']}")

print(
    f"Primary Key Status : "
    f"{validation_result['primary_key']['status']}"
)

print("\nColumns Having Null Values:")

display(validation_result["nulls"])

In [0]:

# Column-wise replacement values
replace_dict = {
    "Amount": 0,
    "Method": "Cash"
}

for col_name, replace_value in replace_dict.items():

    silver_df = silver_df.withColumn(
        col_name,

        F.when(
            F.col(col_name).isNull(),
            F.lit(replace_value)
        ).otherwise(F.col(col_name))
    )

display(silver_df)

Handling Duplicates

Deleting Duplicates

In [0]:
# #Drop Duplicates using hash key
print(f"Before Count: {silver_df.count()}")
silver_df=silver_df.dropDuplicates(["Patientid"])
print(f"After Count: {silver_df.count()}")


Standlize the Data

Standardized text by converting the first character of each record to uppercase.

In [0]:
silver_df = standardize_string_columns(silver_df)
display(silver_df)

In [0]:
#Here rounding off the Amount  and concat with indian rupee symbol
silver_df=silver_df.withColumn("Amount",
        F.round(F.col("Amount"), 2)
    )

display(silver_df)

In [0]:
%sql
SELECT distinct Type FROM `helathcare_bronze`.`helathcare_bronze`.`ClaimsData` --where Method is not null;

Standardized Method values from 'CC'/'DD' to 'Credit card'/'Debit Card'.

In [0]:

silver_df = silver_df.withColumn(
    "Method",

    F.when(
        F.col("Method") == "Cc",
        "Credit Card"
    ).when(
        F.col("Method") == "Echeck",
        "Electronic Check"
    ).when(
        F.col("Method") == "Copay",
        "Co-payment"
    ).otherwise(F.col("Method"))
)

display(silver_df)

Selecting only requrie columns

In [0]:
silver_df = silver_df.select(
    "Id",
    "Claimid",
    "Chargeid",
    "Patientid",
    "Type",
    "Amount",
    "Method",
    "Fromdate",
    "Todate",
    "Placeofservice",
    "Procdurecode",
    "Units",
    "Departmentid",
    "Notes",
    "Filename"


)

In [0]:
validation_result = run_validations(
    silver_df,
    ["Patientid"]
)

print(f"Duplicate Count : {validation_result['duplicates']}")

print(
    f"Primary Key Status : "
    f"{validation_result['primary_key']['status']}"
)

print("\nColumns Having Null Values:")

display(validation_result["nulls"])


#conclusion
# --in DRIVERS have null values,so fill with seuence so i taken min and max for that increase max valu by 1 --max S99999871,min-S99911728
# ---PREFIX have null based on gender column need to fill mr or mis in perfix
#---FIPS have nulls values , need to fill with county name having filps code
#

In [0]:
display(silver_df)

In [0]:
#column rename to amount to Clims_amount
silver_df = silver_df.withColumnRenamed("Amount","Cliams_amount")

In [0]:
# # # Drop table if exists to avoid metadata mismatch
# spark.sql(f"DROP TABLE IF EXISTS helathcare_silver.{silver_schema}.SL_ClaimsData")
# silver_df.write.format("delta") \
#     .option("delta.enableChangeDataFeed", "true") \
#     .option("mergeSchema", "true") \
#     .mode("append") \
#     .saveAsTable(f"helathcare_silver.{silver_schema}.SL_ClaimsData")

In [0]:
#create tempview
silver_df.createOrReplaceTempView(
    "source_claims"
)


In [0]:
%sql
select count(*) from helathcare_silver.helathcare_silver.SL_ClaimsData tgt

In [0]:
claims_count=silver_df.count()

In [0]:
%sql
MERGE INTO helathcare_silver.helathcare_silver.SL_ClaimsData tgt

USING source_claims src

ON tgt.patientid = src.patientid AND tgt.Claimid = src.claimid and tgt.Id = src.Id

WHEN MATCHED THEN
UPDATE SET *

WHEN NOT MATCHED THEN
INSERT *

In [0]:
end_time = datetime.now()

duration_seconds = int(
    (end_time - start_time).total_seconds()
)

print(duration_seconds)

In [0]:
from pyspark.sql.types import LongType
from datetime import datetime

# Get Workflow Run ID
try:
    run_id = dbutils.jobs.taskContext().taskRunId()
except:
    run_id = f"MANUAL_{datetime.now().strftime('%Y%m%d%H%M%S')}"

target_table = "helathcare_silver.helathcare_silver.SL_ClaimsData"

# Get metadata
notebook_name, table_name, layer = get_audit_metadata(target_table)

status = "SUCCESS"
error_message = None
record_count = 0

In [0]:
# Duplicate Check

duplicate_check_status = (
    "PASS"
    if validation_result["duplicates"] == 0
    else "FAIL"
)

# Primary Key Check

primary_key_status = (
    validation_result["primary_key"]["status"]
)

# Null Check

null_count = (
    validation_result["nulls"]
    .agg(F.sum("null_count"))
    .collect()[0][0]
)

null_check_status = (
    "PASS"
    if null_count == 0
    else "FAIL"
)

# Standardization

standardization_status = "PASS"

In [0]:
end_time = datetime.now()

write_audit(
    target_table=target_table,
    run_id=run_id,
    record_count=claims_count,
    start_time=start_time,
    end_time=end_time,
    status=status,
    duplicate_check_status=duplicate_check_status,
    primary_key_status=primary_key_status,
    null_check_status=null_check_status,
    standardization_status=standardization_status,
    error_message=error_message
)